# Hướng Dẫn Chạy Notebook Dành Riêng Cho KAGGLE

1. Upload thư mục Data lên Kaggle Dataset.\n2. Bấm Add Data bên góc phải để chèn vào Notebook.\n3. Chạy Run All. Model đã cài sẵn MF1 Score và Early Stopping.

In [ ]:
# Cài đặt thư viện wandb và scikit-learn
!pip install wandb scikit-learn -q

import os
# Tự động chèn và cấu hình API Key cho Weights & Biases
os.environ['WANDB_API_KEY'] = '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'
os.environ['WANDB_SILENT'] = 'true'


In [ ]:
import wandb
import random
from datetime import datetime
current_time = datetime.now().strftime("%Y%m%d-%H%M%S")

run = wandb.init(
    entity="giabao240806-fpt-university",
    project="Đồng cam mất ngủ",
    config={
        "learning_rate_pretrain": 1e-4,
        "learning_rate_finetune": 1e-5,
        "architecture": "DeepSleepNet",
        "dataset": "Sleep-EDF",
        "epochs_pretrain": 100,
        "epochs_finetune": 100,
        "batch_size" : 32,
        "early_stopping_patience": 20
    },
    name=f"Vũ khí tối thượng 3.0 (MF1 Update)",
    save_code=True
)


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score, cohen_kappa_score
import warnings
warnings.filterwarnings('ignore')
import os

# ====================================================================
# CẤU HÌNH ĐƯỜNG DẪN KAGGLE
# ====================================================================
BASE_DATA_DIR = '/kaggle/input/sleep-edf-processed/'
CHANNEL = 'Fpz-Cz'  # Bạn có thể đổi thành 'Pz-Oz'
DATA_TYPE = 'SC_Data'  # Bạn có thể đổi thành 'ST_Data'

DATA_DIR = os.path.join(BASE_DATA_DIR, CHANNEL, DATA_TYPE)
SAVE_DIR = '/kaggle/working/SLEEP_MODELS/'
os.makedirs(SAVE_DIR, exist_ok=True)

# Hyperparameters
BATCH_SIZE_PRETRAIN = 128
BATCH_SIZE_FINETUNE = 32
SEQ_LEN = 20
EPOCHS_PRETRAIN = 100
EPOCHS_FINETUNE = 100
LR_PRETRAIN = 1e-4
LR_FINETUNE = 1e-5
EARLY_STOPPING_PATIENCE = 20
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", DEVICE)


In [ ]:
class DeepSleepNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=5):
        super(DeepSleepNet, self).__init__()
        
        self.branch1 = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=50, stride=6, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=8, stride=8),
            nn.Dropout(0.5),
            nn.ZeroPad1d((3, 4)),
            nn.Conv1d(64, 128, kernel_size=8, stride=1, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.ZeroPad1d((3, 4)),
            nn.Conv1d(128, 128, kernel_size=8, stride=1, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.ZeroPad1d((3, 4)),
            nn.Conv1d(128, 128, kernel_size=8, stride=1, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4)
        )
        
        self.branch2 = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=400, stride=50, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4),
            nn.Dropout(0.5),
            nn.ZeroPad1d((2, 3)),
            nn.Conv1d(64, 128, kernel_size=6, stride=1, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.ZeroPad1d((2, 3)),
            nn.Conv1d(128, 128, kernel_size=6, stride=1, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.ZeroPad1d((2, 3)),
            nn.Conv1d(128, 128, kernel_size=6, stride=1, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2)
        )
        
        cnn_out_features = 2688
        self.dropout_cnn = nn.Dropout(0.5)
        self.pretrain_classifier = nn.Linear(cnn_out_features, num_classes)
        
        hidden_size = 512
        self.bilstm = nn.LSTM(
            input_size=cnn_out_features, hidden_size=hidden_size, 
            num_layers=2, batch_first=True, bidirectional=True, dropout=0.5
        )
        
        self.shortcut = nn.Linear(cnn_out_features, hidden_size * 2)
        self.finetune_classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(hidden_size * 2, num_classes)
        )

    def freeze_cnn(self):
        for param in self.branch1.parameters():
            param.requires_grad = False
        for param in self.branch2.parameters():
            param.requires_grad = False
            
    def _cnn_feature_extraction(self, x):
        out1 = self.branch1(x)
        out2 = self.branch2(x)
        out1 = out1.view(out1.size(0), -1)
        out2 = out2.view(out2.size(0), -1)
        cnn_features = torch.cat((out1, out2), dim=1)
        cnn_features = self.dropout_cnn(cnn_features)
        return cnn_features

    def forward(self, x, mode="finetune"):
        if mode == "pretrain":
            if x.dim() == 4:
                x = x.squeeze(1) 
            cnn_features = self._cnn_feature_extraction(x)
            return self.pretrain_classifier(cnn_features)
            
        elif mode == "finetune":
            B, Seq_Len, C, L = x.shape
            x = x.view(B * Seq_Len, C, L)
            cnn_features = self._cnn_feature_extraction(x)
            seq_features = cnn_features.view(B, Seq_Len, -1)
            lstm_out, _ = self.bilstm(seq_features)
            shortcut_out = self.shortcut(seq_features)
            combined_features = lstm_out + shortcut_out
            return self.finetune_classifier(combined_features)


In [ ]:
class SingleEpochDataset(Dataset):
    def __init__(self, file_paths, oversample=True):
        self.x = []
        self.y = []
        
        for p in file_paths:
            data = np.load(p)
            self.x.append(data['x'])
            self.y.append(data['y'])
            
        if not self.x:
            self.x = np.empty((0, 1, 3000))
            self.y = np.empty((0,))
            return
            
        self.x = np.concatenate(self.x, axis=0)
        self.y = np.concatenate(self.y, axis=0)
        
        if oversample and len(self.y) > 0:
            classes, counts = np.unique(self.y, return_counts=True)
            max_count = np.max(counts)
            
            x_oversampled, y_oversampled = [], []
            for cls in classes:
                idx = np.where(self.y == cls)[0]
                idx_resampled = np.random.choice(idx, max_count, replace=True)
                x_oversampled.append(self.x[idx_resampled])
                y_oversampled.append(self.y[idx_resampled])
                
            self.x = np.concatenate(x_oversampled, axis=0)
            self.y = np.concatenate(y_oversampled, axis=0)
            
    def __len__(self):
        return len(self.y)
        
    def __getitem__(self, idx):
        return torch.FloatTensor(self.x[idx]), torch.LongTensor([self.y[idx]])[0]


class SequenceDataset(Dataset):
    def __init__(self, file_paths, seq_len=20, stride=20):
        self.x = []
        self.y = []
        
        for p in file_paths:
            data = np.load(p)
            x_subj = data['x']
            y_subj = data['y']
            
            num_epochs = len(y_subj)
            for i in range(0, num_epochs - seq_len + 1, stride):
                self.x.append(x_subj[i : i+seq_len])
                self.y.append(y_subj[i : i+seq_len])
                
        if self.x:
            self.x = np.array(self.x)
            self.y = np.array(self.y)
        else:
            self.x = np.empty((0, seq_len, 1, 3000))
            self.y = np.empty((0, seq_len))
        
    def __len__(self):
        return len(self.y)
        
    def __getitem__(self, idx):
        return torch.FloatTensor(self.x[idx]), torch.LongTensor(self.y[idx])


In [ ]:
def train_step(model, dataloader, optimizer, criterion, mode):
    model.train()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(inputs, mode=mode)
        
        if mode == "finetune":
            outputs = outputs.view(-1, outputs.size(-1))
            labels = labels.view(-1)
            
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
    acc = 100. * correct / total
    mf1 = 100. * f1_score(all_labels, all_preds, average='macro')
    kappa = cohen_kappa_score(all_labels, all_preds)
    return total_loss / len(dataloader), acc, mf1, kappa

def val_step(model, dataloader, criterion, mode):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs, mode=mode)
            
            if mode == "finetune":
                outputs = outputs.view(-1, outputs.size(-1))
                labels = labels.view(-1)
                
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    acc = 100. * correct / total
    mf1 = 100. * f1_score(all_labels, all_preds, average='macro')
    kappa = cohen_kappa_score(all_labels, all_preds)
    return total_loss / len(dataloader), acc, mf1, kappa


In [ ]:
def main():
    all_files = glob.glob(os.path.join(DATA_DIR, "*.npz"))
    if len(all_files) == 0:
        print(f"Chưa có data trong {DATA_DIR}. Vui lòng kiểm tra lại cấu trúc thư mục Kaggle Dataset!")
        return

    print("Khởi tạo K-Fold...")
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    train_idx, val_idx = next(kf.split(all_files))
    
    train_files = [all_files[i] for i in train_idx]
    val_files = [all_files[i] for i in val_idx]
    
    model = DeepSleepNet().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    
    # -------------------------------------
    # BƯỚC 1: PRE-TRAINING
    # -------------------------------------
    print("\n=== BẮT ĐẦU PRE-TRAINING (1-STEP) ===")
    
    train_ds_1 = SingleEpochDataset(train_files, oversample=True)
    val_ds_1 = SingleEpochDataset(val_files, oversample=False)
    
    train_loader_1 = DataLoader(train_ds_1, batch_size=BATCH_SIZE_PRETRAIN, shuffle=True)
    val_loader_1 = DataLoader(val_ds_1, batch_size=BATCH_SIZE_PRETRAIN, shuffle=False)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_PRETRAIN, weight_decay=1e-4)
    
    best_val_mf1_1 = 0
    patience_counter_1 = 0
    
    for epoch in range(EPOCHS_PRETRAIN):
        train_loss, train_acc, train_mf1, train_kappa = train_step(model, train_loader_1, optimizer, criterion, mode="pretrain")
        val_loss, val_acc, val_mf1, val_kappa = val_step(model, val_loader_1, criterion, mode="pretrain")
        
        wandb.log({"epoch": epoch+1, 
                   "pretrain_train_loss": train_loss, "pretrain_val_f1": val_mf1, 
                   "pretrain_val_kappa": val_kappa, "pretrain_val_acc": val_acc})
                   
        print(f"Epoch [{epoch+1}/{EPOCHS_PRETRAIN}] Pretrain - Loss: {train_loss:.4f} - Val Acc: {val_acc:.2f}% - Val MF1: {val_mf1:.2f}%")
        
        # Early Stopping & Checkpoint (Dựa trên MF1 thay vì Accuracy)
        if val_mf1 > best_val_mf1_1:
            best_val_mf1_1 = val_mf1
            patience_counter_1 = 0
            torch.save(model.state_dict(), os.path.join(SAVE_DIR, "deepsleepnet_pretrain_best.pth"))
        else:
            patience_counter_1 += 1
            if patience_counter_1 >= EARLY_STOPPING_PATIENCE:
                print(f"Early Stopping kích hoạt ở Epoch {epoch+1} (Pretrain) do MF1 không tiến bộ!")
                break
                
    # -------------------------------------
    # BƯỚC 2: FINE-TUNING
    # -------------------------------------
    print("\n=== BẮT ĐẦU FINE-TUNING (2-STEP) ===")
    
    # Nạp lại trọng số xịn nhất của Step 1 trước khi sang Step 2
    model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "deepsleepnet_pretrain_best.pth")))
    model.freeze_cnn()
    
    train_ds_2 = SequenceDataset(train_files, seq_len=SEQ_LEN, stride=5) # Sửa OOM: trượt 5 nấc
    val_ds_2 = SequenceDataset(val_files, seq_len=SEQ_LEN, stride=SEQ_LEN)
    
    train_loader_2 = DataLoader(train_ds_2, batch_size=BATCH_SIZE_FINETUNE, shuffle=True)
    val_loader_2 = DataLoader(val_ds_2, batch_size=BATCH_SIZE_FINETUNE, shuffle=False)
    
    optimizer2 = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_FINETUNE)
    
    best_val_mf1_2 = 0
    best_val_acc_2 = 0
    best_val_kappa_2 = 0
    patience_counter_2 = 0
    
    for epoch in range(EPOCHS_FINETUNE):
        train_loss, train_acc, train_mf1, train_kappa = train_step(model, train_loader_2, optimizer2, criterion, mode="finetune")
        val_loss, val_acc, val_mf1, val_kappa = val_step(model, val_loader_2, criterion, mode="finetune")
        
        # Ghi log đúng luật WandB để chuẩn hóa
        wandb.log({"epoch": epoch+1, 
                   "train_loss": train_loss, "val_f1": val_mf1,
                   "val_kappa": val_kappa, "val_acc": val_acc})
                   
        print(f"Epoch [{epoch+1}/{EPOCHS_FINETUNE}] Finetune - Loss: {train_loss:.4f} - Val Acc: {val_acc:.2f}% - Val MF1: {val_mf1:.2f}%")
        
        # Early Stopping & Checkpoint
        if val_mf1 > best_val_mf1_2:
            best_val_mf1_2 = val_mf1
            best_val_acc_2 = val_acc
            best_val_kappa_2 = val_kappa
            patience_counter_2 = 0
            torch.save(model.state_dict(), os.path.join(SAVE_DIR, "deepsleepnet_finetune_best.pth"))
            print("Đã lưu model tốt nhất dựa trên MF1 Score!")
        else:
            patience_counter_2 += 1
            if patience_counter_2 >= EARLY_STOPPING_PATIENCE:
                print(f"Early Stopping kích hoạt ở Epoch {epoch+1} (Finetune) do MF1 không tiến bộ!")
                break
                
    # Ghi log kết quả Final Fold cho nhóm tổng hợp
    wandb.log({"final_fold_acc": best_val_acc_2, 
               "final_fold_f1": best_val_mf1_2, 
               "final_fold_kappa": best_val_kappa_2})
            
    wandb.finish()
    print("✅ HOÀN TẤT TRAINING!")

main()
